In [2]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=601)  # AI4I 2020 Predictive Maintenance
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)
print("Raw shape:", df_raw.shape)

df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]
rename_map = {
    "Type": "type",
    "Air temperature": "air_temp_k",
    "Process temperature": "process_temp_k",
    "Rotational speed": "rpm",
    "Torque": "torque_nm",
    "Tool wear": "tool_wear_min",
    "Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]

print("\nMissing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))

df.to_csv("../data/ai4i_clean.csv", index=False)
print("\nSaved", df.shape, "to data/ai4i_clean.csv")
df.head()

Raw shape: (10000, 12)

Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64

Saved (10000, 7) to data/ai4i_clean.csv


,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "9a94ea5e-07bd-42b7-b7d6-58b18d46274e"
RESOURCE_GROUP = "rg-project1-hasiba0045"
WORKSPACE = "mlw-project1-hasiba0045"

ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

csv_asset = Data(
    name="ai4i-maintenance-csv",
    version="1",
    type=AssetTypes.URI_FILE,
    path="../data/ai4i_clean.csv",
    description="AI4I 2020 predictive maintenance, cleaned: 6 features + machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

Connected to: mlw-project1-hasiba0045


HttpResponseError: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "984c0342e25d4277bfb026c7a94c1dd6",
        "request": "e08f433fff4deda0"
    }
}Type: Environment
Info: {
    "value": "centralindia"
}Type: Location
Info: {
    "value": "centralindia"
}Type: Time
Inf

In [5]:
import os, shutil
from azure.ai.ml import MLClient, automl, Input
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

# 1. Connect
ml_client = MLClient(DefaultAzureCredential(),
                     "9a94ea5e-07bd-42b7-b7d6-58b18d46274e",
                     "rg-project1-hasiba0045",
                     "mlw-project1-hasiba0045")
print("Connected to:", ml_client.workspace_name)

# 2. Table asset (sirf agar pehle se na ho)
try:
    t = ml_client.data.get("ai4i-maintenance-table", version="1")
    print("Table asset already exists:", t.name, t.type)
except Exception:
    mltable_dir = "/tmp/ai4i_mltable"
    os.makedirs(mltable_dir, exist_ok=True)
    shutil.copy("../data/ai4i_clean.csv", mltable_dir)
    with open(f"{mltable_dir}/MLTable", "w") as f:
        f.write("""paths:
  - file: ./ai4i_clean.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
""")
    t = ml_client.data.create_or_update(Data(
        name="ai4i-maintenance-table", version="1", type=AssetTypes.MLTABLE,
        path=mltable_dir, description="AI4I 2020 cleaned, Table asset for Automated ML"))
    print("Registered:", t.name, "version", t.version, "type", t.type)

# 3. AutoML job submit (KNN only)
job = automl.classification(
    compute="cc-project1",
    experiment_name="knn-predictive-maintenance-automl",
    training_data=Input(type="mltable", path="azureml:ai4i-maintenance-table:1"),
    target_column_name="machine_failure",
    primary_metric="AUC_weighted",
    n_cross_validations=5,
    enable_model_explainability=False,
)
job.display_name = "knn-automl-hasiba0045"
job.set_limits(timeout_minutes=30, trial_timeout_minutes=15,
               max_trials=10, enable_early_termination=True)
job.set_training(allowed_training_algorithms=["KNN"],
                 enable_vote_ensemble=False, enable_stack_ensemble=False)
submitted = ml_client.jobs.create_or_update(job)
print("AutoML job submitted:", submitted.name)
print("Open in studio:", submitted.studio_url)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading ai4i_mltable (0.3 MBs): 100%|██████████| 295096/295096 [00:00<00:00, 24823455.87it/s]




Connected to: mlw-project1-hasiba0045
Registered: ai4i-maintenance-table version 1 type mltable
AutoML job submitted: goofy_peach_fh2cjk03lp
Open in studio: https://ml.azure.com/runs/goofy_peach_fh2cjk03lp?wsid=/subscriptions/9a94ea5e-07bd-42b7-b7d6-58b18d46274e/resourcegroups/rg-project1-hasiba0045/workspaces/mlw-project1-hasiba0045&tid=c1741e86-9905-4b62-a136-d7406a597304


In [6]:
import os, pandas as pd
from azure.ai.ml import MLClient, automl, Input
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

ml_client = MLClient(DefaultAzureCredential(),
                     "9a94ea5e-07bd-42b7-b7d6-58b18d46274e",
                     "rg-project1-hasiba0045",
                     "mlw-project1-hasiba0045")

# KNN needs dense numeric data: encode product quality as ordinal L=0, M=1, H=2
df = pd.read_csv("../data/ai4i_clean.csv")
df["type"] = df["type"].map({"L": 0, "M": 1, "H": 2})
mltable_dir = "/tmp/ai4i_mltable_numeric"
os.makedirs(mltable_dir, exist_ok=True)
df.to_csv(f"{mltable_dir}/ai4i_numeric.csv", index=False)
with open(f"{mltable_dir}/MLTable", "w") as f:
    f.write("""paths:
  - file: ./ai4i_numeric.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
""")
t = ml_client.data.create_or_update(Data(
    name="ai4i-maintenance-table", version="2", type=AssetTypes.MLTABLE, path=mltable_dir,
    description="AI4I cleaned, type encoded L=0 M=1 H=2 (dense numeric for AutoML KNN)"))
print("Registered:", t.name, "version", t.version)

# AutoML job v2
job = automl.classification(
    compute="cc-project1",
    experiment_name="knn-predictive-maintenance-automl",
    training_data=Input(type="mltable", path="azureml:ai4i-maintenance-table:2"),
    target_column_name="machine_failure",
    primary_metric="AUC_weighted",
    n_cross_validations=5,
    enable_model_explainability=False,
)
job.display_name = "knn-automl-hasiba0045-v2"
job.set_featurization(mode="custom", column_name_and_types={"type": "Numeric"})
job.set_limits(timeout_minutes=30, trial_timeout_minutes=15,
               max_trials=10, enable_early_termination=True)
job.set_training(allowed_training_algorithms=["KNN"],
                 enable_vote_ensemble=False, enable_stack_ensemble=False)
submitted = ml_client.jobs.create_or_update(job)
print("AutoML job submitted:", submitted.name)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading ai4i_mltable_numeric (0.3 MBs): 100%|██████████| 295098/295098 [00:00<00:00, 5196422.68it/s]




Registered: ai4i-maintenance-table version 2
AutoML job submitted: ivory_arch_zhftkvkrlv
